# RNN serialization, inference and monitoring

**Learning objective:** Save/reload a recurrent model and quantify a simple feature-drift signal.

This notebook is part of the TensorFlow/Keras learning track. It is designed to be read top-to-bottom: intuition → shapes → mathematics → TensorFlow implementation → observed result → interpretation.

> GitHub renders the committed executed output as a static learning artifact. Clone the repository and rerun it in Jupyter/VS Code for live experimentation.


In [1]:
import os, warnings, random
from pathlib import Path
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

print("TensorFlow:", tf.__version__)
print("Keras:", tf.keras.__version__ if hasattr(tf.keras, "__version__") else "bundled with TensorFlow")
print("Execution device(s):", [d.device_type for d in tf.config.list_logical_devices()])


2026-09-21 07:14:02.617467: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1789974842.633737    3234 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1789974842.638136    3234 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


TensorFlow: 2.18.1
Keras: 3.15.1
Execution device(s): ['CPU']


2026-09-21 07:14:04.302922: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [2]:
model=tf.keras.Sequential([tf.keras.layers.Input((6,1)),tf.keras.layers.GRU(4),tf.keras.layers.Dense(1)])
model.compile(optimizer="adam",loss="mse")
sample=tf.random.normal((1,6,1),seed=SEED)
_=model(sample)
path=Path("RNN")/"artifacts_demo_rnn.keras"; model.save(path)
restored=tf.keras.models.load_model(path)
a=model.predict(sample,verbose=0); b=restored.predict(sample,verbose=0)
print("saved:",path,"reload prediction identical:",np.allclose(a,b))


saved: RNN/artifacts_demo_rnn.keras reload prediction identical: True


In [3]:
rng=np.random.default_rng(SEED); reference=rng.normal(0,1,1000); production=rng.normal(0.7,1.2,1000)
drift=(production.mean()-reference.mean())/reference.std()
print("reference mean/std:",reference.mean().round(3),reference.std().round(3)); print("production mean/std:",production.mean().round(3),production.std().round(3)); print("standardized mean shift:",round(float(drift),3))


reference mean/std: -0.029 0.989
production mean/std: 0.602 1.217
standardized mean shift: 0.638


Production monitoring needs both data signals (missingness, distribution shift, sequence length) and outcome signals (error when labels arrive). A saved model without an input contract and monitoring plan is not an operational system.
